In [51]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from PIL import Image
import cv2
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from matplotlib import pyplot as plt, font_manager, rcParams


# Plot font configuration
FONT_PATH = "./fonts/nyk Ngayogyan Jejeg-Apri Nugroho-aksaradinusantara/nyk Ngayogyan Jejeg.ttf"
font_manager.fontManager.addfont(FONT_PATH)
rcParams["font.family"] = font_manager.FontProperties(fname=FONT_PATH).get_name()

torch.cuda.empty_cache()

In [52]:
NGLEGENA = [
    ("ꦲ","ha"), ("ꦤ","na"), ("ꦕ","ca"), ("ꦫ","ra"), ("ꦏ","ka"),
    ("ꦢ","da"), ("ꦠ","ta"), ("ꦱ","sa"), ("ꦮ","wa"), ("ꦭ","la"),
    ("ꦥ","pa"), ("ꦝ","dha"), ("ꦗ","ja"), ("ꦪ","ya"), ("ꦚ","nya"),
    ("ꦩ","ma"), ("ꦒ","ga"), ("ꦧ","ba"), ("ꦛ","tha"), ("ꦔ","nga"),
]

char_list = [c[0] for c in NGLEGENA]
char2idx = {c: i+1 for i, c in enumerate(char_list)}
idx2char = {i+1: c for i, c in enumerate(char_list)}
NUM_CLASSES = len(char_list) + 1


In [53]:
class CNNBiLSTM(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, 3, 1, 1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.Conv2d(256, 256, 3, 1, 1),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),

            nn.Conv2d(256, 512, 3, 1, 1),
            nn.BatchNorm2d(512),
            nn.ReLU(),

            nn.MaxPool2d((2, 1)),
            nn.AdaptiveAvgPool2d((1, None))
        )

        self.rnn = nn.LSTM(
            input_size=512, 
            hidden_size=256, 
            num_layers=2, 
            bidirectional=True, 
            batch_first=True
        )

        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.cnn(x) 

        x = x.squeeze(2)
        x = x.permute(0, 2, 1)

        x, _ = self.rnn(x)

        x = self.fc(x)
        return x


In [54]:
class CLAHE(object):
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clahe = cv2.createCLAHE(
            clipLimit=clip_limit,
            tileGridSize=tile_grid_size
        )

    def __call__(self, img):
        img = np.array(img)

        if len(img.shape) == 3:
            img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

        img = self.clahe.apply(img)

        return Image.fromarray(img)


In [55]:
class ResizeByHeight:
    def __init__(self, height, interpolation=Image.BILINEAR):
        self.height = height
        self.interpolation = interpolation

    def __call__(self, img):
        w, h = img.size
        new_w = int(w * self.height / h)
        return img.resize((new_w, self.height), self.interpolation)

class JavaneseOCRDataset(Dataset):
    def __init__(self, csv_path, img_dir, img_height):
        self.df = pd.read_csv(csv_path)
        self.img_dir = img_dir

        self.transform = T.Compose([
            CLAHE(),
            T.Grayscale(1),
            ResizeByHeight(img_height),
            T.ToTensor(),
            T.Normalize(mean=[0.5], std=[0.5])
        ])

    def encode(self, text):
        return torch.tensor(
            [char2idx[c] for c in text if c in char2idx],
            dtype=torch.long
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.img_dir, row["image"])
        image = Image.open(img_path).convert("L")
        image = self.transform(image)

        label = self.encode(row["transcription"])
        label_len = len(label)

        return image, label, label_len


def ctc_collate_fn(batch):
    images, labels, label_lens = zip(*batch)

    widths = [img.shape[-1] for img in images]
    max_width = max(widths)

    max_width = (max_width + 3) // 4 * 4

    padded_images = []
    input_lens = []

    for img, w in zip(images, widths):
        pad_w = max_width - w
        padded_images.append(
            torch.nn.functional.pad(img, (0, pad_w))
        )
        input_lens.append(w // 4)

    images = torch.stack(padded_images)
    labels = torch.cat(labels)
    label_lens = torch.tensor(label_lens, dtype=torch.long)
    input_lens = torch.tensor(input_lens, dtype=torch.long)

    return images, labels, label_lens, input_lens


In [56]:
def levenshtein(a, b):
    n, m = len(a), len(b)
    if n == 0: return m
    if m == 0: return n

    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[i][j] = min(
                dp[i-1][j] + 1,
                dp[i][j-1] + 1,
                dp[i-1][j-1] + cost
            )
    return dp[n][m]

def cer(preds, refs):
    dist, total = 0, 0
    for p, r in zip(preds, refs):
        dist += levenshtein(p, r)
        total += len(r)
    return dist / max(total, 1)

def em(preds, refs):
    errors = 0
    for p, r in zip(preds, refs):
        if p != r:
            errors += 1
    return errors / max(len(refs), 1)


In [59]:
TEST_CSV_PATH = "./test-sample/label.csv"
TEST_IMG_DIR = "./test-sample/image"

MODEL_PATH = "./models/18font-model.pt"



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNBiLSTM(NUM_CLASSES).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=False))
model.eval()


test_ds = JavaneseOCRDataset(csv_path=TEST_CSV_PATH, img_dir=TEST_IMG_DIR, img_height=64)
test_loader = DataLoader(
    test_ds,
    batch_size=16,
    shuffle=False,
    collate_fn=ctc_collate_fn,
    num_workers=0
)


all_preds = []
all_refs = []
idx = 1
for batch_idx, (images, labels, label_lens, input_lens) in enumerate(test_loader):
    images = images.to(device)

    with torch.no_grad():
        logits = model(images)

    preds = logits.argmax(2)

    offset = 0
    for i in range(images.size(0)):
        pred = preds[i]

        label_len = label_lens[i].item()
        label = labels[offset:offset + label_len]
        offset += label_len

        decoded_pred = []
        prev = 0
        for p in pred:
            p = p.item()
            if p != prev and p != 0:
                decoded_pred.append(idx2char[p])
            prev = p
        decoded_pred = "".join(decoded_pred)

        decoded_label = "".join([idx2char[c.item()] for c in label])

        all_preds.append(decoded_pred)
        all_refs.append(decoded_label)

        print(f"{idx} GT: {decoded_label} | Pred: {decoded_pred}")
        idx += 1

cer_value = cer(all_preds, all_refs)
em_value  = em(all_preds, all_refs)

print(f"\nCER: {cer_value:.4f}")
print(f"EM : {1 - em_value:.4f}")

1 GT: ꦮꦒꦧꦩꦚ | Pred: ꦮꦒꦧꦩꦧꦫ
2 GT: ꦒꦛꦫꦚ | Pred: ꦒꦔꦫꦧ
3 GT: ꦚꦛꦔ | Pred: ꦔꦔꦧ
4 GT: ꦢꦪꦔ | Pred: ꦢꦲꦔ
5 GT: ꦫꦏꦮꦫ | Pred: ꦫꦏꦕꦫ
6 GT: ꦛꦗ | Pred: ꦔꦗ
7 GT: ꦠꦒꦫꦝ | Pred: ꦠꦒꦫꦝ
8 GT: ꦮꦢꦚ | Pred: ꦮꦢꦚꦫ
9 GT: ꦔꦥꦫꦭ | Pred: ꦔꦥꦫꦭ
10 GT: ꦒꦢꦤꦒꦕꦢ | Pred: ꦒꦢꦤꦏꦕꦱ
11 GT: ꦗꦫꦛ | Pred: ꦗꦫꦛ
12 GT: ꦝꦱ | Pred: ꦝꦱ
13 GT: ꦕꦢꦮꦚ | Pred: ꦕꦢꦮꦧꦫ
14 GT: ꦛꦤꦢꦩ | Pred: ꦤꦏꦤꦩ
15 GT: ꦥꦠꦩꦗꦚꦏ | Pred: ꦢꦠꦩꦗꦚꦏ
16 GT: ꦱꦏꦩ | Pred: ꦤꦏꦫꦔ
17 GT: ꦲꦫꦝꦭꦱ | Pred: ꦲꦫꦢꦒꦤ
18 GT: ꦫꦠꦥ | Pred: ꦫꦠꦢ
19 GT: ꦤꦩꦝꦧ | Pred: ꦤꦩꦝꦧ
20 GT: ꦔꦏꦱꦢꦢ | Pred: ꦔꦏꦤꦢꦢ
21 GT: ꦲꦢꦥꦪꦱ | Pred: ꦒꦢꦮꦲꦤ
22 GT: ꦤꦛꦝꦏꦕ | Pred: ꦤꦛꦝꦏꦕ
23 GT: ꦥꦔꦤꦥꦕ | Pred: ꦮꦔꦤꦥꦕ
24 GT: ꦒꦧꦝ | Pred: ꦒꦧꦝ

CER: 0.3368
EM : 0.2917
